In [ ]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm

In [ ]:
folderpath = r"C:\Users\DISO-MGR\Downloads\mypersonal file\Data\Data"

In [ ]:
filelist = os.listdir(folderpath)
filelist

In [ ]:
mergedf = pd.DataFrame()
for file in tqdm(filelist):
    #create filepath
    filepath =os.path.join(folderpath,file)

    #read file
    df= pd.read_excel(filepath, skiprows = 5)

    # Rename first two unnamed columns
    df.rename(columns= {"Unnamed: 0" : "Item No,",
                        "Unnamed: 1" : "Account Desp"}, inplace = True)
    #identify last row for df
    last = df["Account Desp"].str.contains("No. of banks",case = False, na = False).argmax()
    data_df = df.iloc[:last]
    #data_df

    # extract data with loop
    resultdf = pd.DataFrame()
    for col in data_df.columns[2:]:
        extdf = pd.DataFrame()
        extdf["Item No,"] = data_df["Item No,"]
        extdf["Account"] = data_df["Account Desp" ]
        extdf["Amount"] = data_df[col]
        extdf["Month"]= col.split(" ")[0]
        extdf["Year"]= col.split(" ")[1]
        resultdf =pd.concat([resultdf,extdf],ignore_index = True)

    #create Month change to numeric date time data type
    resultdf["Month"] = pd.to_datetime(resultdf["Month"],format="%b",errors ="coerce").dt.month
    #resultdf["Month"].values

    #change data type month, year, Amount
    resultdf["Month"]= pd.to_numeric(resultdf["Month"])
    resultdf["Year"]= pd.to_numeric(resultdf["Year"])
    resultdf["Amount"]= pd.to_numeric(resultdf["Amount"],errors ="coerce").fillna(0)

    #file name column added
    resultdf["SourceFile"] = Path(file).name

    #merge data files
    mergedf = pd.concat([mergedf,resultdf])

In [ ]:
#export excel
mergedf.to_excel("ChawSuSuSoe_Final_Commercial_Bank_Consolidated.xlsx",index =False)
print(" Consolidated files was successfully exported")

In [ ]:
#create Pivot table
result_pivot = mergedf.pivot_table(index = "SourceFile", values= "Amount", aggfunc = "sum",margins = True, margins_name="Grand Total")
result_pivot

In [ ]:
#export pivot excel
result_pivot.reset_index().to_excel("ChawSuSuSoe_Final_Commercial_Bank_ConsolidatedPivot.xlsx",index = False)
print(" Consolidated pivotdata was successfully exported")

In [ ]:
pip freeze > requi rements.txt